# Inférence du LoRA Iris-3B

Ce notebook transforme un prompt en sprite avec le LoRA entraîné. Il choisit le checkpoint, démarre le moteur à la demande, affiche l’avancement puis enregistre l’image.

**Utilisation :** exécute les cellules de haut en bas. Pour une autre image, modifie la cellule « Prompt » et relance seulement « Générer ». Si le moteur a été arrêté, cette cellule le redémarre automatiquement.

Le modèle et l’encodeur texte sont quantifiés en FP8. Toutes leurs couches sont déchargées vers la RAM système pour limiter la VRAM ; les transferts ralentissent l’inférence. Le calcul se fait en BF16 et le LoRA est appliqué en FP16.


## 1. Préparer l’environnement

Le kernel doit disposer de PyTorch avec CUDA et du dossier AI Toolkit. La cellule suivante vérifie les petits utilitaires Python et ne les installe que s’ils manquent. Elle n’installe pas PyTorch et ne remplace pas le pilote NVIDIA.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {"requests": "requests", "PIL": "pillow", "dotenv": "python-dotenv"}
missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print("Installation des utilitaires manquants :", ", ".join(missing_packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("Utilitaires Python prêts.")


## 2. Trouver le LoRA et vérifier le GPU

Le run hiérarchique `iris_pixelart_sprite_96_humanoid` est prioritaire ; l’ancien run `iris_pixelart_sprite_96` sert de repli. Pour choisir un autre checkpoint, renseigne `LORA_PATH_OVERRIDE` dans la cellule de code suivante.


In [ ]:
import json
import os
import re
import subprocess
import sys
import time
import uuid
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path
from urllib.parse import quote

import requests
import torch
from dotenv import load_dotenv
from IPython.display import Markdown, display
from PIL import Image

NOTEBOOK_DIR = Path.cwd()
env_candidates = [NOTEBOOK_DIR / ".env", NOTEBOOK_DIR / "training" / ".env", NOTEBOOK_DIR.parent / ".env", Path("/content/.env")]
ENV_FILE = next((path for path in env_candidates if path.is_file()), None)
if ENV_FILE is not None:
    load_dotenv(dotenv_path=ENV_FILE)

work_candidates = [
    NOTEBOOK_DIR / "iris_sprite_lora_work",
    NOTEBOOK_DIR / "training" / "iris_sprite_lora_work",
    NOTEBOOK_DIR,
    NOTEBOOK_DIR.parent / "iris_sprite_lora_work",
    NOTEBOOK_DIR.parent / "training" / "iris_sprite_lora_work",
    NOTEBOOK_DIR / "notebook" / "training" / "iris_sprite_lora_work",
    Path("/content/iris_sprite_lora"),
]
WORK_DIR = next((path for path in work_candidates if (path / "ai-toolkit" / "run.py").is_file()), None)
if WORK_DIR is None:
    raise FileNotFoundError(
        "Projet AI Toolkit introuvable. Ouvre le notebook depuis le dossier du projet "
        "ou place iris_sprite_lora_work à côté."
    )

AI_TOOLKIT_DIR = WORK_DIR / "ai-toolkit"
OUTPUT_ROOT = WORK_DIR / "output"
INFERENCE_DIR = WORK_DIR / "inference_iris3b_sprite_lora"
JOB_FOLDER = INFERENCE_DIR / "engine"
JOB_CONFIG = INFERENCE_DIR / "inference_engine.json"
ENGINE_LOG_PATH = INFERENCE_DIR / "engine.log"
ENDPOINT_PATH = JOB_FOLDER / "engine.json"
GENERATED_DIR = INFERENCE_DIR / "generated"
for directory in (INFERENCE_DIR, JOB_FOLDER, GENERATED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

LORA_PATH_OVERRIDE = None  # Ex. Path("/chemin/vers/mon_lora.safetensors")
RUN_PRIORITY = ["iris_pixelart_sprite_96_humanoid", "iris_pixelart_sprite_96"]


def find_checkpoint(run_name):
    run_dir = OUTPUT_ROOT / run_name
    final_path = run_dir / f"{run_name}.safetensors"
    if final_path.is_file():
        return final_path
    candidates = []
    for path in run_dir.glob(f"{run_name}_*.safetensors"):
        match = re.fullmatch(re.escape(run_name) + r"_(\d+)", path.stem)
        if match:
            candidates.append((int(match.group(1)), path.stat().st_mtime, path))
    return max(candidates)[2] if candidates else None


if LORA_PATH_OVERRIDE:
    LORA_PATH = Path(LORA_PATH_OVERRIDE).expanduser().resolve()
    SELECTED_RUN = "fichier choisi manuellement"
    if not LORA_PATH.is_file():
        raise FileNotFoundError(f"Checkpoint LoRA introuvable : {LORA_PATH}")
else:
    LORA_PATH = None
    SELECTED_RUN = None
    for run_name in RUN_PRIORITY:
        checkpoint = find_checkpoint(run_name)
        if checkpoint is not None:
            LORA_PATH, SELECTED_RUN = checkpoint, run_name
            break

if LORA_PATH is None:
    existing = sorted(str(path.relative_to(OUTPUT_ROOT)) for path in OUTPUT_ROOT.rglob("*.safetensors")) if OUTPUT_ROOT.exists() else []
    details = "\n".join(existing[:20]) or "aucun fichier .safetensors trouvé"
    raise FileNotFoundError(
        f"Aucun LoRA trouvé dans {OUTPUT_ROOT}.\n{details}\n"
        "Lance l’entraînement ou renseigne LORA_PATH_OVERRIDE."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA n’est pas disponible dans ce kernel. Active un runtime NVIDIA et "
        "sélectionne l’environnement Python qui contient PyTorch CUDA."
    )

free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print(f"Projet : {WORK_DIR}")
print(f"Run sélectionné : {SELECTED_RUN}")
print(f"Checkpoint : {LORA_PATH} ({LORA_PATH.stat().st_size / 1024**2:.1f} MiB)")
print(f"GPU : {torch.cuda.get_device_name(0)} — {total_bytes / 1024**3:.1f} Go")
print(f"VRAM libre : {free_bytes / 1024**3:.2f} Go")
print("HF_TOKEN : défini" if os.environ.get("HF_TOKEN") else "HF_TOKEN : absent (le cache Hugging Face peut suffire)")
if free_bytes < 1 * 1024**3:
    print("Attention : moins de 1 Go de VRAM est libre ; les autres processus GPU peuvent encore gêner.")


## 3. Prompt

Modifie `CATEGORY_PATH` et `DESCRIPTION`. Utilise les catégories apprises : `humanoid → character → playable`, `humanoid → enemy`, ou `creature`. Garde 96 × 96 pour commencer.


In [ ]:
CATEGORY_PATH = ["humanoid", "character", "playable"]
DESCRIPTION = "a playable humanoid character standing with a knife"

SEED = 42
LORA_STRENGTH = 1.0
WIDTH = 96
HEIGHT = 96
STEPS = 30
GUIDANCE_SCALE = 3.0
NEGATIVE_PROMPT = ""

MODEL_CONFIG = {
    "arch": "iris3b",
    "name_or_path": "speridlabs/iris-3b",
    "dtype": "bf16",
    "quantize": True,
    "qtype": "qfloat8",
    "quantize_te": True,
    "qtype_te": "qfloat8",
    "low_vram": True,
    "layer_offloading": True,
    "layer_offloading_transformer_percent": 1.0,
    "layer_offloading_text_encoder_percent": 1.0,
    "lora_mode": "hook",
}


## 4. Moteur d’inférence

Le serveur démarre au premier appel de génération. S’il est déjà actif, le notebook le réutilise. Cette cellule prépare la gestion du serveur ; elle ne charge pas encore le modèle.


In [ ]:
ENGINE_PROC = globals().get("ENGINE_PROC")
ENGINE_PROCESS_OWNED = globals().get("ENGINE_PROCESS_OWNED", False)
ENGINE_LOG_HANDLE = globals().get("ENGINE_LOG_HANDLE")
ENGINE_ENDPOINT = None
ENGINE_URL = None
ENGINE_HEADERS = None


def read_engine_log_tail(max_chars=5000):
    if not ENGINE_LOG_PATH.is_file():
        return ""
    tail = ENGINE_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-max_chars:]
    token = os.environ.get("HF_TOKEN")
    return tail.replace(token, "<HF_TOKEN masqué>") if token else tail


def endpoint_is_healthy(endpoint):
    if not isinstance(endpoint, dict) or not all(endpoint.get(key) for key in ("host", "port", "token")):
        return False
    try:
        response = requests.get(f"http://{endpoint['host']}:{endpoint['port']}/health", timeout=3)
        return response.ok and response.json().get("ok") is True
    except (requests.RequestException, ValueError, KeyError, TypeError):
        return False


def stop_owned_process():
    global ENGINE_PROC, ENGINE_PROCESS_OWNED, ENGINE_LOG_HANDLE
    if ENGINE_PROC is not None and ENGINE_PROC.poll() is None:
        ENGINE_PROC.terminate()
        try:
            ENGINE_PROC.wait(timeout=30)
        except subprocess.TimeoutExpired:
            ENGINE_PROC.kill()
            ENGINE_PROC.wait(timeout=10)
    ENGINE_PROCESS_OWNED = False
    if ENGINE_LOG_HANDLE is not None:
        ENGINE_LOG_HANDLE.close()
        ENGINE_LOG_HANDLE = None


def ensure_engine():
    global ENGINE_ENDPOINT, ENGINE_URL, ENGINE_HEADERS
    global ENGINE_PROC, ENGINE_PROCESS_OWNED, ENGINE_LOG_HANDLE

    endpoint = None
    if ENDPOINT_PATH.is_file():
        try:
            endpoint = json.loads(ENDPOINT_PATH.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            pass

    if endpoint_is_healthy(endpoint):
        ENGINE_ENDPOINT = endpoint
        ENGINE_URL = f"http://{endpoint['host']}:{endpoint['port']}"
        ENGINE_HEADERS = {"X-Engine-Token": endpoint["token"]}
        ENGINE_PROCESS_OWNED = bool(
            ENGINE_PROCESS_OWNED and ENGINE_PROC is not None and ENGINE_PROC.poll() is None
        )
        print("Moteur déjà actif — réutilisation du serveur.")
        return

    if ENGINE_PROCESS_OWNED:
        stop_owned_process()
    ENDPOINT_PATH.unlink(missing_ok=True)

    config = {
        "job": "extension",
        "config": {
            "name": "iris_sprite_inference_engine",
            "process": [{
                "type": "InferenceEngine",
                "device": "cuda:0",
                "engine": {"job_folder": str(JOB_FOLDER)},
            }],
        },
    }
    JOB_CONFIG.write_text(json.dumps(config, indent=2), encoding="utf-8")
    if ENGINE_LOG_HANDLE is not None:
        try:
            ENGINE_LOG_HANDLE.close()
        except Exception:
            pass
    ENGINE_LOG_HANDLE = open(ENGINE_LOG_PATH, "w", encoding="utf-8")

    try:
        ENGINE_PROC = subprocess.Popen(
            [sys.executable, "run.py", str(JOB_CONFIG)],
            cwd=AI_TOOLKIT_DIR,
            env=os.environ.copy(),
            stdout=ENGINE_LOG_HANDLE,
            stderr=subprocess.STDOUT,
        )
        ENGINE_PROCESS_OWNED = True
        deadline = time.monotonic() + 180
        while time.monotonic() < deadline:
            if ENGINE_PROC.poll() is not None:
                ENGINE_LOG_HANDLE.flush()
                raise RuntimeError("Le moteur s’est arrêté au démarrage.\n" + read_engine_log_tail())
            if ENDPOINT_PATH.is_file():
                try:
                    candidate = json.loads(ENDPOINT_PATH.read_text(encoding="utf-8"))
                except (OSError, json.JSONDecodeError):
                    candidate = None
                if endpoint_is_healthy(candidate):
                    ENGINE_ENDPOINT = candidate
                    ENGINE_URL = f"http://{candidate['host']}:{candidate['port']}"
                    ENGINE_HEADERS = {"X-Engine-Token": candidate["token"]}
                    print("Moteur prêt. Le modèle sera chargé à la génération.")
                    return
            time.sleep(1)
        raise TimeoutError("Le moteur ne répond pas après 180 secondes.\n" + read_engine_log_tail())
    except Exception:
        stop_owned_process()
        ENDPOINT_PATH.unlink(missing_ok=True)
        raise


## 5. Générer un sprite

La cellule ci-dessous démarre le moteur si nécessaire, affiche le prompt envoyé et montre si le modèle charge ou si la génération avance. Le résultat est enregistré dans le dossier `inference_iris3b_sprite_lora/generated`.


In [ ]:
ensure_engine()

if not CATEGORY_PATH or not DESCRIPTION.strip():
    raise ValueError("Renseigne une catégorie et une description avant de générer.")
if WIDTH <= 0 or HEIGHT <= 0 or WIDTH % 16 or HEIGHT % 16:
    raise ValueError("La largeur et la hauteur doivent être positives et multiples de 16.")
if STEPS <= 0:
    raise ValueError("STEPS doit être supérieur à zéro.")

category_tokens = []
for value in CATEGORY_PATH + ["sprite", "pixel art", f"{WIDTH}x{HEIGHT}"]:
    value = str(value).strip()
    if value and value.lower() not in {"unknown", "not_applicable"} and value not in category_tokens:
        category_tokens.append(value)
prompt = ", ".join(category_tokens) + "\n\n" + DESCRIPTION.strip()
print("Prompt envoyé :\n" + prompt)
print(f"Réglages : {WIDTH} × {HEIGHT}, {STEPS} étapes, graine {SEED}, force LoRA {LORA_STRENGTH}")

request_body = {
    "model": {
        **MODEL_CONFIG,
        "loras": [{"path": str(LORA_PATH), "strength": float(LORA_STRENGTH)}],
    },
    "sample": {
        "prompt": prompt,
        "negative_prompt": NEGATIVE_PROMPT,
        "width": int(WIDTH),
        "height": int(HEIGHT),
        "num_inference_steps": int(STEPS),
        "guidance_scale": float(GUIDANCE_SCALE),
        "seed": int(SEED),
        "output_ext": "png",
    },
    "wait": True,
}

status_view = display(Markdown("**État :** demande envoyée. Chargement du modèle si nécessaire…"), display_id=True)
request_error = None
with ThreadPoolExecutor(max_workers=1) as executor:
    future = executor.submit(
        requests.post,
        f"{ENGINE_URL}/generate",
        headers=ENGINE_HEADERS,
        json=request_body,
        timeout=(15, 3600),
    )
    last_status = None
    while not future.done():
        try:
            health = requests.get(f"{ENGINE_URL}/health", timeout=3).json()
            current = health.get("current") or {}
            step = int(current.get("step") or 0)
            total = int(current.get("total_steps") or STEPS)
            if current and step:
                message = f"Génération en cours : étape {step}/{total}."
            elif current:
                message = "Chargement du modèle et encodage du prompt ; le premier passage peut prendre quelques minutes."
            else:
                message = "Demande en attente dans le moteur…"
            if message != last_status:
                status_view.update(Markdown(f"**État :** {message}"))
                last_status = message
        except (requests.RequestException, ValueError, AttributeError):
            pass
        time.sleep(2)
    try:
        response = future.result()
    except requests.RequestException as error:
        request_error = error

if request_error is not None:
    raise RuntimeError(
        f"Connexion au moteur interrompue : {request_error}\n"
        f"Journal : {ENGINE_LOG_PATH}\n{read_engine_log_tail()}"
    ) from request_error

try:
    payload = response.json()
except ValueError as error:
    raise RuntimeError(
        f"Le moteur a renvoyé une réponse non JSON (HTTP {response.status_code}).\n"
        f"Journal : {ENGINE_LOG_PATH}\n{read_engine_log_tail()}"
    ) from error

if response.status_code != 200 or payload.get("status") != "done":
    message = str(payload.get("error") or f"HTTP {response.status_code}")
    if "outofmemory" in message.lower() or "out of memory" in message.lower():
        message += "\n\nL’offload complet est déjà actif. Ferme les autres processus GPU et vérifie la RAM système."
    raise RuntimeError(f"Échec de la génération : {message}\nJournal : {ENGINE_LOG_PATH}\n{read_engine_log_tail()}")

results = payload.get("results") or []
if not results:
    raise RuntimeError("Le moteur a terminé sans renvoyer d’image.")
result = results[0]
relative_path = result.get("relpath") or Path(result["path"]).name
image_response = requests.get(
    f"{ENGINE_URL}/outputs/{quote(relative_path, safe='/')}",
    headers=ENGINE_HEADERS,
    timeout=60,
)
if not image_response.ok:
    raise RuntimeError(f"Image générée, mais téléchargement impossible : HTTP {image_response.status_code}")

filename = f"iris3b_{datetime.now():%Y%m%d_%H%M%S}_seed{result.get('seed', SEED)}_{uuid.uuid4().hex[:6]}.png"
OUTPUT_PATH = GENERATED_DIR / filename
OUTPUT_PATH.write_bytes(image_response.content)
with Image.open(OUTPUT_PATH) as generated_image:
    display(generated_image.convert("RGB").resize((WIDTH * 4, HEIGHT * 4), Image.Resampling.NEAREST))
status_view.update(Markdown("**État :** génération terminée."))
print(f"Image enregistrée : {OUTPUT_PATH}")
print(f"Temps : {result.get('seconds', '?')} s — graine : {result.get('seed', SEED)} — étapes : {result.get('steps', STEPS)}")


## 6. Libérer la mémoire

Exécute cette cellule à la fin. Si ce notebook a démarré le serveur, il l’arrête. S’il en a réutilisé un autre, il décharge le modèle et conserve le serveur. Tu peux ensuite relancer « Générer » : le moteur redémarrera au besoin.


In [ ]:
if "ENGINE_PROC" not in globals():
    print("Le moteur n’a pas été démarré dans cette session.")
elif ENGINE_PROCESS_OWNED:
    stop_owned_process()
    print("Serveur arrêté et VRAM libérée.")
elif ENGINE_ENDPOINT and endpoint_is_healthy(ENGINE_ENDPOINT):
    response = requests.post(f"{ENGINE_URL}/unload", headers=ENGINE_HEADERS, timeout=60)
    if response.ok:
        print("Modèle déchargé. Le serveur existant reste actif.")
    else:
        print(f"Déchargement impossible : HTTP {response.status_code}.")
else:
    print("Aucun serveur actif à nettoyer.")
